# 차량 번호판 탐지 모델 학습 및 추론 (YOLO26n)

## 프로젝트 개요
차량 이미지와 영상에서 번호판 영역을 탐지하는 객체 탐지 모델을 학습하고, 학습된 가중치로 이미지·영상 추론을 수행합니다.

- **모델:** Ultralytics YOLO26n
- **클래스:** `LP` (License Plate)
- **학습 환경:** Google Colab GPU (Tesla T4)
- **학습 설정:** 50 epochs, image size 640, batch size 8
- **추론:** 테스트 이미지 및 차량 영상

> 데이터셋과 영상 파일은 용량 및 라이선스 문제로 저장소에 포함하지 않습니다. 직접 준비한 파일을 Colab에 업로드해 실행합니다.

## 1. 환경 설정

Google Colab에서 GPU 런타임을 선택한 뒤 필요한 라이브러리를 설치합니다.

In [ ]:
!pip install -U ultralytics

## 2. 데이터셋 준비

Roboflow YOLO 형식으로 내보낸 데이터셋 ZIP 파일을 업로드합니다. 압축을 푼 뒤 `data.yaml`이 `/content/data.yaml`에 있는지 확인합니다.

In [ ]:
import zipfile
from pathlib import Path

zip_name = next(iter(uploaded.keys()))
zip_path = Path("/content") / zip_name

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

data_yaml = Path("/content/data.yaml")
if not data_yaml.exists():
    candidates = list(Path("/content").rglob("data.yaml"))
    if not candidates:
        raise FileNotFoundError("압축 해제 경로에서 data.yaml을 찾을 수 없습니다.")
    data_yaml = candidates[0]

print("Dataset config:", data_yaml)

In [ ]:
import zipfile
from pathlib import Path

zip_path = next(Path("/content").glob("*.zip"))

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

data_yaml = Path("/content/data.yaml")
if not data_yaml.exists():
    candidates = list(Path("/content").rglob("data.yaml"))
    if not candidates:
        raise FileNotFoundError("압축 해제 경로에서 data.yaml을 찾을 수 없습니다.")
    data_yaml = candidates[0]

print("Dataset config:", data_yaml)

## 3. YOLO26n 모델 학습

사전 학습된 `yolo26n.pt`를 불러와 번호판 탐지 데이터셋으로 파인튜닝합니다. 학습 결과는 `runs/detect/license_plate_yolo26/`에 저장됩니다.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

train_results = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=8,
    patience=20,
    workers=2,
    name="license_plate_yolo26"
)

## 4. 학습 결과 확인

학습 과정에서 생성된 결과 그래프와 CSV를 확인합니다. `best.pt`는 검증 성능이 가장 좋았던 가중치입니다.

In [ ]:
from pathlib import Path
from IPython.display import display, Image

run_dir = Path("/content/runs/detect/license_plate_yolo26")
best_weights = run_dir / "weights" / "best.pt"

if not best_weights.exists():
    raise FileNotFoundError(f"학습 가중치를 찾을 수 없습니다: {best_weights}")

results_image = run_dir / "results.png"
if results_image.exists():
    display(Image(filename=str(results_image)))

### 검증 결과

학습 로그의 best.pt 검증 결과:

| 지표 | 결과 |
|---|---:|
| Precision | 0.997 |
| Recall | 0.993 |
| mAP@0.5 | 0.995 |
| mAP@0.5:0.95 | 0.871 |

※ 위 수치는 학습에 사용된 검증 데이터 기준이며, 별도의 독립 테스트셋 성능으로 해석하지 않습니다.

## 5. 테스트 이미지 추론

테스트 이미지 폴더를 대상으로 번호판 영역을 탐지하고 결과 이미지를 저장합니다.

In [ ]:
best_model = YOLO(str(best_weights))

image_results = best_model.predict(
    source="/content/test/images",
    save=True,
    conf=0.25
)

print("Image inference completed.")

## 6. 차량 영상 추론

영상 파일을 업로드한 뒤 탐지합니다. 원본 실행 기록에서 `conf=0.05`, `imgsz=960` 설정으로 532프레임 영상 추론 결과가 생성된 것을 확인했습니다.

In [ ]:
uploaded_video = files.upload()  # 차량 영상 파일 업로드
video_path = next(
    (Path("/content") / name for name in uploaded_video.keys()),
    None
)

if video_path is None or not video_path.exists():
    raise FileNotFoundError("업로드한 영상 파일을 찾을 수 없습니다.")

video_results = best_model.predict(
    source=str(video_path),
    save=True,
    conf=0.05,
    imgsz=960,
    stream=True
)

# stream=True로 프레임 결과를 순차 처리하여 메모리 사용량을 줄입니다.
frame_count = sum(1 for _ in video_results)
print(f"Video inference completed: {frame_count} frames")

## 7. 결과물

학습 및 추론 후 Colab의 다음 경로에서 결과를 확인할 수 있습니다.

- 학습 가중치: `runs/detect/license_plate_yolo26/weights/best.pt`
- 학습 그래프: `runs/detect/license_plate_yolo26/results.png`
- 학습 지표 CSV: `runs/detect/license_plate_yolo26/results.csv`
- 테스트 이미지 추론: `runs/detect/predict/`
- 영상 추론: `runs/detect/predict*/`

가중치와 결과 영상은 저장소에 무조건 포함하기보다 GitHub Release 또는 별도 공유 링크로 관리하는 것을 권장합니다.